# Урок 20. Электронные таблицы: расчёты и анализ данных

10 класс · III четверть

[⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-10/index.ipynb) · [← Урок 19](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-10/urok-19.ipynb) · [Урок 21 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-10/urok-21.ipynb)

---

Возврат к таблицам на новом уровне: именованные диапазоны, ВПР, СЧЁТЕСЛИМН, сводные таблицы. Когда задача решается формулой за минуту, а когда кодом — за час, и наоборот.

In [ ]:
#@title 🚀 Регистрация { display-mode: "form" }
#@markdown Впиши свои данные и запусти ячейку (Shift+Enter).
#@markdown Если заводил секрет `INFORMATIKA` и включил к нему доступ
#@markdown (🔑 на панели слева) — поля можно оставить пустыми.
#@markdown
#@markdown Colab спросит разрешение на доступ к аккаунту — нажми
#@markdown «Разрешить». Так проверка понимает, чья это работа.
ФИО = "" #@param {type:"string"}
#@markdown Класс — с буквой, например 10А
Класс = "" #@param {type:"string"}

import importlib, urllib.request
urllib.request.urlretrieve("https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/lib/schoolinf.py", "schoolinf.py")
import schoolinf as si
importlib.reload(si)
si.start(lesson="10-20", name=ФИО, klass=Класс)

## Разбираемся

### Таблицы на взрослом уровне

Формулы и условные функции вы знаете. Сегодня — три инструмента,
которые отделяют «умею складывать столбик» от «умею анализировать
данные»: подстановка, многокритериальный отбор и сводные таблицы.

Это ровно то, чем в обычной работе занимаются экономисты, логисты,
маркетологи и половина офиса вообще.

### Подстановка: ВПР

Задача: в одной таблице заказы с кодами товаров, в другой —
справочник «код → название, цена». Нужно подтянуть цену к заказам.

```
=ВПР(искомое; таблица; номер_столбца; ЛОЖЬ)
```

| Аргумент | Что значит |
|---|---|
| искомое | что ищем (код товара) |
| таблица | где ищем (диапазон справочника) |
| номер_столбца | какой столбец справочника вернуть |
| ЛОЖЬ | требовать точное совпадение |

Три правила, без которых ВПР не работает:

1. **Искать можно только по первому столбцу** указанного диапазона.
   Если код лежит во втором — диапазон надо начинать со второго.
2. **Последний аргумент — всегда ЛОЖЬ** (или 0). ИСТИНА означает
   приблизительный поиск и молча подставляет ближайшее меньшее.
3. **Диапазон справочника закрепляют долларами** — иначе при
   копировании он уедет.

Если совпадения нет, ВПР возвращает `#Н/Д`. Это не ошибка, а ответ:
«такого кода в справочнике нет». Оборачивают `=ЕСЛИОШИБКА(ВПР(...);
"нет в справочнике")`.

В современных таблицах есть ПРОСМОТРX (XLOOKUP) — он удобнее
и ищет в любом столбце. Но на экзамене и в старых файлах
встречается именно ВПР.

### Многокритериальный отбор

Функции с окончанием `МН` принимают сколько угодно пар
«диапазон — условие»:

```
=СЧЁТЕСЛИМН(A:A; "Москва"; C:C; ">1000")
=СУММЕСЛИМН(D:D; A:A; "Москва"; C:C; ">1000")
```

Условия соединяются только через «и». Нужно «или» — складывают
несколько функций или заводят вспомогательный столбец.

> Порядок аргументов у `СУММЕСЛИМН` **обратный** по сравнению
> с `СУММЕСЛИ`: сначала что суммировать, потом условия.
> Это классический источник ошибок.

### Сводная таблица

Сводная таблица отвечает на вопрос «сколько чего по категориям»
без единой формулы. Вы перетаскиваете поля в три области:

| Область | Что кладут | Пример |
|---|---|---|
| строки | по чему группировать | город |
| столбцы | второй разрез | месяц |
| значения | что считать | сумма заказов |

Получается таблица «город × месяц», в клетках — суммы. То же самое
в коде — группировка по ключу, которую мы разберём ниже.

Главное достоинство сводной: она пересобирается за секунды.
Хотите не суммы, а количество — меняете одну настройку.

### Когда таблица перестаёт справляться

| Признак | Что делать |
|---|---|
| больше ~100 000 строк | база данных или код |
| одни и те же действия каждую неделю | скрипт |
| нужен нестандартный алгоритм | код |
| файл открывают пять человек сразу | база данных |
| результат надо объяснить и показать | таблица, она нагляднее |

Правильный ответ почти всегда «и то, и другое»: считать кодом,
показывать таблицей.

## Смотрим, как это работает

Всё, что делает таблица, мы повторим в коде — так видно механику.

### Пример 1. Данные: заказы и справочник

In [ ]:
справочник = [
    [101, "Хлеб", 45],
    [102, "Молоко", 89],
    [103, "Сыр", 420],
    [104, "Кофе", 350],
]

заказы = [
    ["Москва", 101, 3],
    ["Москва", 103, 1],
    ["Казань", 102, 5],
    ["Москва", 104, 2],
    ["Казань", 101, 7],
    ["Омск", 103, 2],
    ["Казань", 104, 1],
]

print("Справочник:")
for строка in справочник:
    print("  ", строка)
print("Заказы:", len(заказы), "строк")

### Пример 2. ВПР своими руками

In [ ]:
def впр(код, справочник, столбец):
    for строка in справочник:
        if строка[0] == код:
            return строка[столбец]
    return "#Н/Д"


print("Код 103 →", впр(103, справочник, 1), "по цене", впр(103, справочник, 2))
print("Код 999 →", впр(999, справочник, 1))

Видно, почему ВПР ищет только по первому столбцу: сравнение идёт
именно со `строка[0]`. И видно, откуда берётся `#Н/Д` — это ответ
«не нашлось», а не поломка.

### Пример 3. Подтягиваем цены к заказам

In [ ]:
print(f"{'город':<8} {'товар':<8} {'кол-во':>6} {'цена':>6} {'сумма':>8}")
итого = 0

for город, код, количество in заказы:
    название = впр(код, справочник, 1)
    цена = впр(код, справочник, 2)
    сумма = цена * количество
    итого += сумма
    print(f"{город:<8} {название:<8} {количество:>6} {цена:>6} {сумма:>8}")

print(f"{'ИТОГО':<8} {'':<8} {'':>6} {'':>6} {итого:>8}")

### Пример 4. Многокритериальный отбор

In [ ]:
москва_дорогие = 0
сумма_москва_дорогие = 0

for город, код, количество in заказы:
    цена = впр(код, справочник, 2)
    if город == "Москва" and цена > 100:
        москва_дорогие += 1
        сумма_москва_дорогие += цена * количество

print("Заказов из Москвы дороже 100 за штуку:", москва_дорогие)
print("На сумму:", сумма_москва_дорогие)
print()
print('В таблице: =СЧЁТЕСЛИМН(A:A; "Москва"; E:E; ">100")')

### Пример 5. Сводная таблица: сумма по городам

In [ ]:
по_городам = {}

for город, код, количество in заказы:
    цена = впр(код, справочник, 2)
    по_городам[город] = по_городам.get(город, 0) + цена * количество

print(f"{'город':<10} {'сумма':>8}")
for город in sorted(по_городам):
    print(f"{город:<10} {по_городам[город]:>8}")

Словарь здесь работает ровно как область «строки» сводной таблицы:
ключ — категория, значение — итог. Подробно словари разберём
на отдельном уроке, пока достаточно `.get(ключ, 0)` —
«значение или ноль, если ключа ещё нет».

### Пример 6. Сводная в два разреза

Теперь как настоящая сводная: города в строках, товары в столбцах.

In [ ]:
товары = [строка[1] for строка in справочник]
сводная = {}

for город, код, количество in заказы:
    название = впр(код, справочник, 1)
    сводная[(город, название)] = сводная.get((город, название), 0) + количество

print(f"{'':<10}" + "".join(f"{т:>9}" for т in товары))
for город in sorted(по_городам):
    строка = f"{город:<10}"
    for товар in товары:
        строка += f"{сводная.get((город, товар), 0):>9}"
    print(строка)

Ключ словаря — пара «город, товар». Это и есть пересечение строки
и столбца сводной таблицы.

### Пример 7. Проверка: сходится ли итог

In [ ]:
из_сводной = sum(по_городам.values())

print("Итог по строкам заказов:", итого)
print("Итог по сводной:        ", из_сводной)
print("Сходится:", итого == из_сводной)

Сводная, которая не сходится с исходными данными, — первый признак
того, что диапазон захватил не то. Проверять итог обязательно.

## Пробуем сами

Справочник приходит списком `[код, название, цена]`, заказы —
списком `[город, код, количество]`.

### Задача 1. По какому столбцу ищет ВПР

В каком столбце указанного диапазона ВПР ищет значение?

In [ ]:
#@title 🧩 Задача 1. Как работает ВПР { display-mode: "form" }
#@markdown Выбери ответ
где_ищет = "выбери ответ" #@param ["выбери ответ", "первому столбцу", "последнему столбцу", "любому столбцу"]

si.ответ("1", где_ищет, "cdcd65a04dcdbaaa",
         hint="Поэтому диапазон и начинают со столбца с кодами.")

### Задача 2. Как соединяются условия

Несколько условий в `СЧЁТЕСЛИМН` соединяются логической операцией…

In [ ]:
#@title 🧩 Задача 2. Соединение условий { display-mode: "form" }
#@markdown Выбери операцию
операция = "выбери ответ" #@param ["выбери ответ", "и", "или", "исключающее или"]

si.ответ("2", операция, "dcf3b793bf6f4a8f",
         hint="Строка должна подойти под все условия сразу.")

### Задача 3. ВПР в коде

Функция ищет код в справочнике и возвращает значение нужного
столбца. Если кода нет, вернуть строку `"#Н/Д"`.

In [ ]:
def подстановка(код, справочник, столбец):
    return ...

In [ ]:
si.check("3", подстановка, [
    ((102, [[101, "Хлеб", 45], [102, "Молоко", 89]], 1), "Молоко"),
    ((102, [[101, "Хлеб", 45], [102, "Молоко", 89]], 2), 89),
    ((999, [[101, "Хлеб", 45]], 1), "#Н/Д"),
    ((101, [], 1), "#Н/Д"),
])

### Задача 4. Отбор по двум условиям

Функция возвращает количество заказов из заданного города,
в которых количество товара не меньше порога.

In [ ]:
def счётеслимн(заказы, город, порог):
    return ...

In [ ]:
si.check("4", счётеслимн, [
    (([["Москва", 101, 3], ["Москва", 102, 1], ["Казань", 101, 5]], "Москва", 2), 1),
    (([["Москва", 101, 3]], "Москва", 10), 0),
    (([["Казань", 101, 5], ["Казань", 102, 7]], "Казань", 5), 2),
    (([], "Москва", 1), 0),
])

### Задача 5. Сводка по городам

Функция возвращает словарь «город → суммарное количество товаров».

Это область «строки» сводной таблицы.

In [ ]:
def сводка(заказы):
    return ...

In [ ]:
si.check("5", сводка, [
    ([["Москва", 101, 3], ["Казань", 102, 5], ["Москва", 103, 1]],
     {"Москва": 4, "Казань": 5}),
    ([["Омск", 101, 2]], {"Омск": 2}),
    ([], {}),
])

### Задача 6. Стоимость заказов

Функция получает заказы и справочник и возвращает общую стоимость:
для каждого заказа цену из справочника умножаем на количество.

Заказы с неизвестным кодом пропускаем.

In [ ]:
def стоимость(заказы, справочник):
    return ...

In [ ]:
si.check("6", стоимость, [
    (([["Москва", 101, 3]], [[101, "Хлеб", 45]]), 135),
    (([["Москва", 101, 2], ["Казань", 102, 1]],
      [[101, "Хлеб", 45], [102, "Молоко", 89]]), 179),
    (([["Москва", 999, 3]], [[101, "Хлеб", 45]]), 0),
    (([], [[101, "Хлеб", 45]]), 0),
])

## Домашнее задание

### Домашнее задание 1. Сводная таблица по своим данным

Возьмите таблицу хотя бы на 200 строк с двумя категориальными
столбцами (город и месяц, предмет и класс, категория и магазин)
и постройте сводную таблицу: категории в строках, второй разрез
в столбцах, сумма в значениях.

Ответьте по ней на три вопроса, а один из ответов проверьте
формулой `СУММЕСЛИМН`. Покажите файл учителю.

### Домашнее задание 2. Сводка с ценами

Функция возвращает словарь «город → стоимость заказов этого города».
Цену берите из справочника.

In [ ]:
def сводка_по_деньгам(заказы, справочник):
    return ...

In [ ]:
si.check("дз2", сводка_по_деньгам, [
    (([["Москва", 101, 2], ["Казань", 101, 1]], [[101, "Хлеб", 45]]),
     {"Москва": 90, "Казань": 45}),
    (([["Омск", 102, 3]], [[102, "Молоко", 89]]), {"Омск": 267}),
    (([], [[101, "Хлеб", 45]]), {}),
])

### Домашнее задание 3. Как называется инструмент

Как называется инструмент таблиц, который группирует данные
по категориям без единой формулы?

In [ ]:
#@title 🏠 Домашнее 3. Инструмент группировки { display-mode: "form" }
#@markdown Выбери ответ
инструмент = "выбери ответ" #@param ["выбери ответ", "сводная таблица", "условное форматирование", "проверка данных"]

si.ответ("дз3", инструмент, "8ad3d2eda9609017",
         hint="Поля перетаскивают в строки, столбцы и значения.")

---

### Любопытно

В 2020 году британское агентство здравоохранения потеряло
16 тысяч результатов тестов на коронавирус. Причина оказалась
обидной: данные собирали в файле старого формата Excel, где лист
вмещает 65 536 строк. Всё, что не помещалось, просто не попадало
в отчёт — молча.

Тысячи контактов заболевших не отследили. Это ровно тот случай
из таблицы «когда таблица перестаёт справляться»: данных стало
больше, чем инструмент способен удержать, — а предупреждения
никто не увидел.

---

## Отчёт по уроку

Запусти ячейку ниже, когда решишь задачи. Результат уйдёт учителю автоматически.

In [ ]:
si.report()

---

[← Урок 19](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-10/urok-19.ipynb) · [⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-10/index.ipynb) · [🏠 Ко всем классам](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/index.ipynb) · [Урок 21 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-10/urok-21.ipynb)